# #L905 自動投票 (SPAT4) v01

楽天競馬で取得した T-10/T-3 オッズの **下落率シグナル** で SPAT4 に自動投票する。

## 動作フロー
1. `data/odds_snapshots/` を `poll_interval_sec` ごとに監視
2. 新規 T-3 ファイルを検出 → 同 race_id の T-10 を探す
3. `(odds_T3 - odds_T10) / odds_T10 * 100` を計算
4. 設定閾値以下の買い目を抽出
   - 単勝: `tan_change_rate_max` 以下 (デフォルト -40%)
   - 馬連: `uma_change_rate_max` 以下 (デフォルト -50%)
5. オッズ範囲・1レース上限・1日上限を適用してフィルタ
6. SPAT4 で投票 (DRY-RUN モードでは画面操作のみ、確定はしない)

## 設定
すべての閾値・上限は `config/spat4_credentials.json` で変更可能。
gitignore 済みなので機密情報はリポジトリに入らない。

| キー | デフォルト | 説明 |
|---|---|---|
| `thresholds.T_START` | T10 | 比較元 T-label |
| `thresholds.T_END` | T3 | 比較先 T-label (= 発火タイミング) |
| `thresholds.tan_change_rate_max` | -30.0 | 単勝の change_rate(%) 上限 (これ以下で買う) — #L909 集計の最適 -32% / 期待回収率 138% を踏まえた暫定 |
| `thresholds.uma_change_rate_max` | -30.0 | 馬連の change_rate(%) 上限 (これ以下で買う) — #L909 集計の最適 -32% / 期待回収率 193% を踏まえた暫定 |
| `betting.stake_yen` | 100 | 1 買い目あたりの賭け金 |
| `betting.dry_run` | true | 投票を確定しない (ログ出力のみ) |
| `betting.max_bets_per_race_tan` | 1 | 単勝の 1 レースあたり最大買い目数 (下落率最大のものを残す) |
| `betting.max_bets_per_race_uma` | 10 | 馬連の 1 レースあたり最大買い目数 (下落率最大のものを残す) |
| `betting.max_total_bets_per_day` | 500 | 1 日累計上限 |
| `betting.min_odds_to_buy` | 1.5 | この単勝オッズ未満は買わない |
| `betting.max_odds_to_buy` | 2000.0 | この単勝オッズ超過は買わない |
| `operation.poll_interval_sec` | 10 | スナップショット監視ポーリング間隔 |
| `operation.data_root` | (自動検出, 最低優先) | データルート。G:/マイドライブ/ → D:/ → C:/ の順に **実在チェック** で自動採用 (= サブPC では config を無視して G ドライブを選ぶ)。明示指定したい場合は `NAR_DATA_ROOT` 環境変数 |

## ⚠ 安全運用
- **DRY-RUN がデフォルト**。`dry_run: false` に手動切替するまで投票は確定されない
- LIVE 運用前に手動で SPAT4 にログインして残高・買い目を確認
- 初日は `stake_yen: 100`、`max_total_bets_per_day: 5` 程度から開始
- 異常検出時は Ctrl+C で即停止可能

In [1]:
import os
import sys
import json
import time
import traceback
import importlib
from datetime import datetime, date
from pathlib import Path

import pandas as pd

# racing-common パッケージへのパス
_RC_DIR = Path('C:/Users/ppny9/workspace/racing-common')
if str(_RC_DIR) not in sys.path:
    sys.path.insert(0, str(_RC_DIR))

from racing_common import spat4
importlib.reload(spat4)   # コード変更を毎回反映

from racing_common.notify import send_email_if_configured


In [2]:
# ════════════════════════════════════════════════════════════
# 設定ロード
# ════════════════════════════════════════════════════════════
NAR_ROOT = Path('C:/Users/ppny9/workspace/nar')
CONFIG_PATH = NAR_ROOT / 'config' / 'spat4_credentials.json'

config = spat4.load_config(CONFIG_PATH)

# 閾値
T_START = config['thresholds']['T_START']
T_END   = config['thresholds']['T_END']
TAN_CHANGE_RATE_MAX = config['thresholds']['tan_change_rate_max']
UMA_CHANGE_RATE_MAX = config['thresholds']['uma_change_rate_max']

# 投票設定
STAKE_YEN              = config['betting']['stake_yen']
DEBUG                  = config['betting'].get('debug', False)
MAX_BETS_PER_RACE_TAN  = config['betting']['max_bets_per_race_tan']
MAX_BETS_PER_RACE_UMA  = config['betting']['max_bets_per_race_uma']
MAX_TOTAL_BETS_PER_DAY = config['betting']['max_total_bets_per_day']

# オッズ範囲フィルタ (式別ごとに別設定)
MIN_ODDS_TAN = config['betting'].get('min_odds_tan', 2.0)
MAX_ODDS_TAN = config['betting'].get('max_odds_tan', 30.0)
MIN_ODDS_UMA = config['betting'].get('min_odds_uma', 30.0)
MAX_ODDS_UMA = config['betting'].get('max_odds_uma', 500.0)

# 運用設定 — data_root はマシン環境を自動判定
def _resolve_data_root() -> Path:
    import os as _os
    env = _os.environ.get('NAR_DATA_ROOT', '')
    if env and Path(env).is_dir():
        return Path(env)
    for p in [
        'G:/マイドライブ/workspace/nar/data',
        'D:/workspace/nar/data',
        'C:/Users/ppny9/workspace/nar/data',
    ]:
        if Path(p).is_dir():
            return Path(p)
    return Path(config.get('operation', {}).get('data_root', 'D:/workspace/nar/data'))


DATA_ROOT          = _resolve_data_root()
POLL_INTERVAL_SEC  = config['operation']['poll_interval_sec']
LOG_DIR            = Path('C:/Users/ppny9/workspace/nar') / config['operation'].get('log_dir', 'logs/')
LOG_DIR.mkdir(parents=True, exist_ok=True)

SNAPSHOT_DIR = DATA_ROOT / 'odds_snapshots'

# ── メール通知用の環境変数を config から展開 ──
_email = config.get('email', {})
if _email.get('from') and _email.get('app_pw') and _email.get('to'):
    os.environ.setdefault('NOTIFY_SMTP_USER', _email['from'])
    os.environ.setdefault('NOTIFY_SMTP_PASS', _email['app_pw'])
    os.environ.setdefault('NOTIFY_EMAIL_TO',  _email['to'])
    os.environ.setdefault('NOTIFY_SMTP_HOST', _email.get('smtp_host', 'smtp.gmail.com'))
    os.environ.setdefault('NOTIFY_SMTP_PORT', str(_email.get('smtp_port', 465)))
    print(f'メール通知: {_email["from"]} → {_email["to"]}')
else:
    print(f'メール通知: 未設定 — 送信スキップ')

print(f'CONFIG_PATH: {CONFIG_PATH}')
print(f'T_START → T_END: {T_START} → {T_END}')
print(f'閾値: 単勝 ≤ {TAN_CHANGE_RATE_MAX}% / 馬連 ≤ {UMA_CHANGE_RATE_MAX}%')
print(f'STAKE: {STAKE_YEN}円')
print(f'上限: 単勝{MAX_BETS_PER_RACE_TAN}点/race  馬連{MAX_BETS_PER_RACE_UMA}点/race  累計{MAX_TOTAL_BETS_PER_DAY}/day')
print(f'オッズ範囲: 単勝 {MIN_ODDS_TAN}〜{MAX_ODDS_TAN}倍  馬連 {MIN_ODDS_UMA}〜{MAX_ODDS_UMA}倍')
print(f'監視dir: {SNAPSHOT_DIR}')


メール通知: anago.fresh.jjaf@gmail.com → anago.fresh.jjaf@gmail.com
CONFIG_PATH: C:\Users\ppny9\workspace\nar\config\spat4_credentials.json
T_START → T_END: T10 → T3
閾値: 単勝 ≤ -30.0% / 馬連 ≤ -30.0%
STAKE: 100円
上限: 単勝1点/race  馬連10点/race  累計500/day
オッズ範囲: 単勝 2.0〜30.0倍  馬連 20.0〜9999.0倍
監視dir: G:\マイドライブ\workspace\nar\data\odds_snapshots


In [3]:
# ════════════════════════════════════════════════════════════
# シグナル抽出ロジック
# ════════════════════════════════════════════════════════════
def list_snapshot_files(directory: Path, race_id: str = None, label: str = None,
                        kind: str = None) -> list[Path]:
    """snapshot ファイルを filter してリスト化。"""
    files = list(directory.glob('*.csv'))
    out = []
    import re
    for p in files:
        m = re.match(r'(\d{12,18})_(T\d+)_(tanfuku|umaren|sanrentan)_', p.name)
        if not m:
            continue
        if race_id and m.group(1) != race_id:
            continue
        if label and m.group(2) != label:
            continue
        if kind and m.group(3) != kind:
            continue
        out.append(p)
    return out


def load_latest_snapshot(race_id: str, label: str, kind: str) -> pd.DataFrame:
    """指定 (race_id, label, kind) の最新スナップショット 1 件を読込み。"""
    files = list_snapshot_files(SNAPSHOT_DIR, race_id=race_id, label=label, kind=kind)
    if not files:
        return pd.DataFrame()
    latest = sorted(files, key=lambda p: p.stat().st_mtime)[-1]
    return pd.read_csv(latest, encoding='utf-8-sig')


def compute_signals_for_race(race_id: str) -> dict:
    """指定 race_id について T_START → T_END の change_rate を計算し、シグナル買い目を返す。

    Returns: {'tan_bets': [...], 'uma_bets': [...], 'race_id': ...}
    """
    bets = {'race_id': race_id, 'tan_bets': [], 'uma_bets': []}

    # ── 単勝 (tanfuku): 2〜30倍 ──
    df_t_start = load_latest_snapshot(race_id, T_START, 'tanfuku')
    df_t_end   = load_latest_snapshot(race_id, T_END,   'tanfuku')
    if not df_t_start.empty and not df_t_end.empty:
        m = df_t_start[['umaban', 'odds_tan']].rename(columns={'odds_tan': 'odds_start'}) \
              .merge(df_t_end[['umaban', 'odds_tan']].rename(columns={'odds_tan': 'odds_end'}),
                     on='umaban')
        m = m.dropna(subset=['odds_start', 'odds_end'])
        m = m[m['odds_start'] > 0]
        m['change_rate'] = (m['odds_end'] - m['odds_start']) / m['odds_start'] * 100
        hits = m[(m['change_rate'] <= TAN_CHANGE_RATE_MAX)
                 & (m['odds_end'] >= MIN_ODDS_TAN)
                 & (m['odds_end'] <= MAX_ODDS_TAN)]
        for _, r in hits.iterrows():
            bets['tan_bets'].append({
                'umaban': int(r['umaban']),
                'odds_start': float(r['odds_start']),
                'odds_end': float(r['odds_end']),
                'change_rate': float(r['change_rate']),
            })

    # ── 馬連 (umaren): 30〜500倍 ──
    df_u_start = load_latest_snapshot(race_id, T_START, 'umaren')
    df_u_end   = load_latest_snapshot(race_id, T_END,   'umaren')
    if not df_u_start.empty and not df_u_end.empty:
        m = df_u_start[['P1', 'P2', 'odds_umaren']].rename(columns={'odds_umaren': 'odds_start'}) \
              .merge(df_u_end[['P1', 'P2', 'odds_umaren']].rename(columns={'odds_umaren': 'odds_end'}),
                     on=['P1', 'P2'])
        m = m.dropna(subset=['odds_start', 'odds_end'])
        m = m[m['odds_start'] > 0]
        m['change_rate'] = (m['odds_end'] - m['odds_start']) / m['odds_start'] * 100
        hits = m[(m['change_rate'] <= UMA_CHANGE_RATE_MAX)
                 & (m['odds_end'] >= MIN_ODDS_UMA)
                 & (m['odds_end'] <= MAX_ODDS_UMA)]
        for _, r in hits.iterrows():
            bets['uma_bets'].append({
                'P1': int(r['P1']),
                'P2': int(r['P2']),
                'odds_start': float(r['odds_start']),
                'odds_end': float(r['odds_end']),
                'change_rate': float(r['change_rate']),
            })

    return bets


In [4]:
# ════════════════════════════════════════════════════════════
# race_id からレース情報 (venue, race_num, race_date) を解決
# ════════════════════════════════════════════════════════════
SHUTSUBA_DIR = DATA_ROOT / 'shutsuba'


def resolve_race_info(race_id: str) -> dict:
    """race_id → {venue, race_num, race_date, place_id (= SPAT4 JOCD)}

    place_id は SPAT4_PLACE_ID マップ (venue名 → JOCD) を優先して取得する。
    マップに存在しない場合のみ race_id[8:10] にフォールバック。
    注: race_id[8:10] は楽天の内部コードであり SPAT4 JOCD とは異なる場合がある。
    例: 門別 race_id[8:10]=36 だが SPAT4 JOCD=30 / 園田 race_id[8:10]=27 だが SPAT4 JOCD=50
    """
    info = {'race_id': race_id}
    if len(race_id) == 12:
        info['race_date'] = f'{race_id[:4]}-{race_id[6:8]}-{race_id[8:10]}'
        info['race_num']  = int(race_id[10:12])
    elif len(race_id) == 18:
        info['race_date'] = f'{race_id[:4]}-{race_id[4:6]}-{race_id[6:8]}'
        info['race_num']  = int(race_id[16:18])
    else:
        info['race_date'] = '?'
        info['race_num']  = 0

    # venue は shutsuba から
    sh_file = SHUTSUBA_DIR / f'{race_id}_shutsuba.csv'
    if sh_file.exists():
        try:
            df_sh = pd.read_csv(sh_file, encoding='utf-8-sig', nrows=1)
            if 'venue' in df_sh.columns and len(df_sh) > 0:
                info['venue'] = str(df_sh['venue'].iloc[0])
            else:
                info['venue'] = '?'
        except Exception:
            info['venue'] = '?'
    else:
        info['venue'] = '?'

    # place_id: SPAT4_PLACE_ID マップを優先 (race_id[8:10] は楽天内部コードで JOCD と異なる)
    spat4_jocd = spat4.venue_to_place_id(info['venue']) if info['venue'] != '?' else None
    if spat4_jocd:
        info['place_id'] = spat4_jocd
    elif len(race_id) == 18:
        info['place_id'] = race_id[8:10]   # フォールバック (マップ未登録場)
    else:
        info['place_id'] = '00'
    return info


In [5]:
# ════════════════════════════════════════════════════════════
# 投票実行ラッパ
# ════════════════════════════════════════════════════════════
import re as _re
from datetime import timedelta

RESULT_DIR = DATA_ROOT / 'results'

class DailyBetCounter:
    def __init__(self, max_per_day: int):
        self.max = max_per_day
        self.date = date.today()
        self.count = 0

    def reset_if_new_day(self):
        if date.today() != self.date:
            print(f'  [INFO] 日付変更 {self.date} → {date.today()}、カウンタリセット')
            self.date = date.today()
            self.count = 0

    def can_bet(self, n: int = 1) -> bool:
        self.reset_if_new_day()
        return self.count + n <= self.max

    def add(self, n: int = 1):
        self.count += n


daily_counter = DailyBetCounter(MAX_TOTAL_BETS_PER_DAY)
pending_hits: list[dict] = []


def _parse_t3_snapshot_dt(t3_path) -> 'datetime | None':
    """T3 ファイル名 ({race_id}_T3_{kind}_{YYYYMMDD}-{HHMMSS}.csv) から snapshot 時刻を取得。"""
    m = _re.search(r'_(\d{8})-(\d{6})\.csv$', str(t3_path))
    if m:
        try:
            return datetime.strptime(m.group(1) + m.group(2), '%Y%m%d%H%M%S')
        except Exception:
            pass
    return None


def _format_bet_notification(race_info: dict, tan_bets: list, uma_bets: list,
                              t_start: str, t_end: str) -> tuple:
    n_tan, n_uma = len(tan_bets), len(uma_bets)
    now = datetime.now()

    # T3 検知時刻 → 推定締め切り (T_END の数値分後)
    snapshot_dt = race_info.get('t3_snapshot_dt')
    t_end_min = int(t_end[1:]) if len(t_end) > 1 and t_end[1:].isdigit() else 3
    if snapshot_dt:
        est_close_dt = snapshot_dt + timedelta(minutes=t_end_min)
        time_str = f' 締切{est_close_dt:%H:%M}'
    else:
        est_close_dt = None
        time_str = ''

    subj = (f'[NAR L905] 買い目 {race_info.get("venue","?")} R{race_info.get("race_num","?")}'
            f' 単勝{n_tan} 馬複{n_uma}{time_str}')

    lines = []
    lines.append(f'通知時刻  : {now:%Y-%m-%d %H:%M:%S}')
    if snapshot_dt:
        lines.append(f'T3検知    : {snapshot_dt:%H:%M:%S}  推定締め切り: {est_close_dt:%H:%M:%S}')
    lines.append(f'race_id : {race_info.get("race_id","")}')
    lines.append(f'venue   : {race_info.get("venue","")}  R{race_info.get("race_num","")}')
    lines.append(f'race_date: {race_info.get("race_date","")}')
    lines.append(f'閾値    : {t_start} → {t_end}  /  単勝 ≤ {TAN_CHANGE_RATE_MAX}%  馬複 ≤ {UMA_CHANGE_RATE_MAX}%')
    lines.append('')

    if tan_bets:
        lines.append(f'■ 単勝 ({n_tan} 点)')
        for b in tan_bets:
            lines.append(f'  馬番 {b["umaban"]:>2}  '
                          f'{t_start}={b["odds_start"]:>7.1f}  →  '
                          f'{t_end}={b["odds_end"]:>7.1f}  '
                          f'({b["change_rate"]:+.2f}%)')
        lines.append('')

    if uma_bets:
        lines.append(f'■ 馬複 ({n_uma} 点)')
        for b in uma_bets:
            lines.append(f'  {b["P1"]:>2}-{b["P2"]:<2}  '
                          f'{t_start}={b["odds_start"]:>7.1f}  →  '
                          f'{t_end}={b["odds_end"]:>7.1f}  '
                          f'({b["change_rate"]:+.2f}%)')
        lines.append('')

    lines.append(f'stake   : {STAKE_YEN}円 × {n_tan + n_uma} 点')
    return subj, '\n'.join(lines)


def check_hit_and_notify(pending: dict) -> bool:
    race_id = pending['race_id']
    result_path = RESULT_DIR / f'{race_id}_result.csv'
    payout_path = RESULT_DIR / f'{race_id}_payout.csv'
    if not result_path.exists() or not payout_path.exists():
        return False
    try:
        df_result = pd.read_csv(result_path, encoding='utf-8-sig')
        df_payout = pd.read_csv(payout_path, encoding='utf-8-sig')
    except Exception as e:
        print(f'  [WARN] 結果ファイル読み込み失敗 {race_id}: {e}')
        return False
    winner_rows = df_result[df_result['rank'] == 1]
    if winner_rows.empty:
        return False
    winner_umaban = int(winner_rows.iloc[0]['umaban'])
    venue = pending['venue']
    rnum  = pending['race_num']
    hits  = []
    total_return = 0
    for b in pending['tan_bets']:
        if b['umaban'] == winner_umaban:
            row = df_payout[df_payout['kenshu'] == '単勝']
            payout = int(row.iloc[0]['payout']) if not row.empty else 0
            hits.append(f'単勝 馬番{b["umaban"]} → {payout}円')
            total_return += payout
    for b in pending['uma_bets']:
        combo = f'{min(b["P1"], b["P2"])}-{max(b["P1"], b["P2"])}'
        for kenshu_name in ['馬連', '馬複']:
            row = df_payout[(df_payout['kenshu'] == kenshu_name) & (df_payout['combo'] == combo)]
            if not row.empty:
                payout = int(row.iloc[0]['payout'])
                hits.append(f'馬複 {combo} → {payout}円')
                total_return += payout
                break
    now = datetime.now()
    if hits:
        subj = f'[NAR L905] 的中！ {venue} R{rnum}  +{total_return}円'
        lines = [f'確認時刻  : {now:%Y-%m-%d %H:%M:%S}',
                 f'{venue} R{rnum}  (race_id={race_id})', '']
        lines += hits
        lines += ['', f'合計払戻: {total_return}円',
                  f'賭け金合計: {STAKE_YEN * (len(pending["tan_bets"]) + len(pending["uma_bets"]))}円']
        body = '\n'.join(lines)
        print(f'  [HIT] {subj}')
        send_email_if_configured(subj, body)
    else:
        print(f'  [MISS] {venue} R{rnum} 不的中 (1着: 馬番{winner_umaban})')
    return True


def execute_bets_for_race(sess: spat4.Spat4Session, driver, race_info: dict, signals: dict) -> dict:
    tan_sorted = sorted(signals['tan_bets'], key=lambda b: b['change_rate'])
    uma_sorted = sorted(signals['uma_bets'], key=lambda b: b['change_rate'])
    tan_bets = tan_sorted[:MAX_BETS_PER_RACE_TAN]
    uma_bets = uma_sorted[:MAX_BETS_PER_RACE_UMA]

    total_to_bet = len(tan_bets) + len(uma_bets)
    if total_to_bet == 0:
        return {'ok': True, 'detail': '買い目なし', 'count': 0}
    if not daily_counter.can_bet(total_to_bet):
        print(f'  [SKIP] 日次上限 {MAX_TOTAL_BETS_PER_DAY} に到達 (現在 {daily_counter.count})')
        return {'ok': True, 'detail': '日次上限', 'count': 0}

    result = sess.place_bets_for_race(driver, race_info, tan_bets, uma_bets, STAKE_YEN)

    # ── メール通知: 投票が実際に確定した場合のみ送信 ──
    if result['ok'] and result['count'] > 0:
        subj, body = _format_bet_notification(race_info, tan_bets, uma_bets, T_START, T_END)
        try:
            sent = send_email_if_configured(subj, body)
            if sent:
                print(f'  [MAIL] 通知送信: {subj}')
        except Exception as e:
            print(f'  [MAIL ERROR] {e}')

    log_path = LOG_DIR / f'L905_bet_log_{date.today():%Y%m%d}.csv'
    log_exists = log_path.exists()
    with open(log_path, 'a', encoding='utf-8-sig') as f:
        if not log_exists:
            f.write('timestamp,race_id,venue,race_num,kenshu,combo,'
                    'odds_start,odds_end,change_rate,stake,result\n')
        ts = datetime.now().isoformat(timespec='seconds')
        for b in tan_bets:
            f.write(f'{ts},{race_info["race_id"]},{race_info["venue"]},{race_info["race_num"]},'
                    f'tan,{b["umaban"]},{b["odds_start"]},{b["odds_end"]},{b["change_rate"]:.2f},'
                    f'{STAKE_YEN},{result["ok"]}\n')
        for b in uma_bets:
            f.write(f'{ts},{race_info["race_id"]},{race_info["venue"]},{race_info["race_num"]},'
                    f'uma,{b["P1"]}-{b["P2"]},{b["odds_start"]},{b["odds_end"]},{b["change_rate"]:.2f},'
                    f'{STAKE_YEN},{result["ok"]}\n')

    if result['ok'] and result['count'] > 0:
        daily_counter.add(result['count'])
        pending_hits.append({
            'race_id':  race_info['race_id'],
            'venue':    race_info.get('venue', '?'),
            'race_num': race_info.get('race_num', '?'),
            'tan_bets': tan_bets,
            'uma_bets': uma_bets,
        })
    return result


In [6]:
# ════════════════════════════════════════════════════════════
# シグナルの単発テスト (実投票なし)
# ════════════════════════════════════════════════════════════
# 動作確認: 既存の race_id の T-10 → T-3 シグナルを計算
def test_signal_for_race(race_id: str):
    info = resolve_race_info(race_id)
    print(f'race_id={race_id}  venue={info["venue"]}  R{info["race_num"]}  date={info["race_date"]}')
    sig = compute_signals_for_race(race_id)
    print(f'  単勝シグナル: {len(sig["tan_bets"])} 件')
    for b in sig['tan_bets']:
        print(f'    馬番{b["umaban"]}: {b["odds_start"]:.1f} → {b["odds_end"]:.1f}  ({b["change_rate"]:+.1f}%)')
    print(f'  馬連シグナル: {len(sig["uma_bets"])} 件')
    for b in sig['uma_bets']:
        print(f'    {b["P1"]}-{b["P2"]}: {b["odds_start"]:.1f} → {b["odds_end"]:.1f}  ({b["change_rate"]:+.1f}%)')


# 既存 race_id のサンプルで確認
sample_ids = [p.name.split('_')[0] for p in SNAPSHOT_DIR.glob('*_T3_*.csv')]
sample_ids = sorted(set(sample_ids))[-5:]   # 最新 5 件
print(f'=== 最新の T-3 を持つ race_id (last 5) ===')
for rid in sample_ids:
    test_signal_for_race(rid)
    print()

=== 最新の T-3 を持つ race_id (last 5) ===
race_id=202606262320050301  venue=笠松  R1  date=2026-06-26
  単勝シグナル: 0 件
  馬連シグナル: 0 件

race_id=202606262320050302  venue=笠松  R2  date=2026-06-26
  単勝シグナル: 0 件
  馬連シグナル: 0 件

race_id=202606262320050304  venue=笠松  R4  date=2026-06-26
  単勝シグナル: 0 件
  馬連シグナル: 1 件
    1-2: 204.9 → 112.4  (-45.1%)

race_id=202606262320050305  venue=笠松  R5  date=2026-06-26
  単勝シグナル: 0 件
  馬連シグナル: 0 件

race_id=202606262320050306  venue=笠松  R6  date=2026-06-26
  単勝シグナル: 0 件
  馬連シグナル: 1 件
    3-4: 683.8 → 457.7  (-33.1%)



In [7]:
# ════════════════════════════════════════════════════════════
# DEBUG ブラウザ確認 (ダミー買い目で確認画面の手前まで)
# ════════════════════════════════════════════════════════════
def debug_run_browser(
    race_info: dict = None,
    tan_umabans: list = None,
    uma_pairs: list = None,
    wait_sec: int = 120,
):
    """DEBUG=True 専用: ダミー買い目で SPAT4 操作を確認画面の手前まで実行。
    passcode 入力・確定ボタンは一切押さない。

    SPAT4 UI フロー (1ステップごとにノートブックへログ出力):
      投票入力クリック
      → 場名選択 → R番選択 → 式別選択 → 通常選択 → 馬番選択 → 金額入力 → 入力決定
      ↑ 買い目ごとに繰り返し
      → 投票内容確認へ (DEBUG ではここで停止)

    Args:
        race_info   : 対象レース情報 dict。
                      必須キー: place_id (JOCD 2桁), race_num, race_date (YYYY-MM-DD), venue
        tan_umabans : 単勝で試す馬番リスト (デフォルト [1])
        uma_pairs   : 馬複で試すペアリスト (デフォルト [(1, 2)])
        wait_sec    : 最後の買い目後、ブラウザを開いたまま待機する秒数 (目視確認用)
    """
    if not DEBUG:
        print('[SKIP] DEBUG=False — config の betting.debug を true にしてから再実行')
        return

    if race_info is None:
        race_info = {
            'race_id': 'DEBUG',
            'venue': '大井',
            'race_num': 1,
            'race_date': datetime.now().strftime('%Y-%m-%d'),
            'place_id': '44',   # 大井 JOCD
        }
    if tan_umabans is None:
        tan_umabans = [1]
    if uma_pairs is None:
        uma_pairs = [(1, 2)]

    dummy_tan = [
        {'umaban': u, 'odds_start': 5.0, 'odds_end': 3.0, 'change_rate': -40.0}
        for u in tan_umabans
    ]
    dummy_uma = [
        {'P1': p1, 'P2': p2, 'odds_start': 10.0, 'odds_end': 6.0, 'change_rate': -40.0}
        for p1, p2 in uma_pairs
    ]

    print(f'[DEBUG] race_info : {race_info}')
    print(f'[DEBUG] 単勝 馬番  : {tan_umabans}')
    print(f'[DEBUG] 馬複 ペア  : {uma_pairs}')
    print(f'[DEBUG] Selenium 起動 ...')

    sess = spat4.Spat4Session(config, headless=False)
    with sess.driver() as driver:
        if not sess.login(driver):
            print('  [FATAL] SPAT4 login 失敗')
            return

        # まとめて投票 (DEBUG フラグ=True のため確認画面の手前で停止)
        result = sess.place_bets_for_race(driver, race_info, dummy_tan, dummy_uma, STAKE_YEN)
        tag = 'OK' if result['ok'] else 'NG'
        print(f'  [{tag}] {result["detail"]}')

        print(f'[DEBUG] {wait_sec} 秒間ブラウザを開いたままにします ...')
        time.sleep(wait_sec)

    print('[DEBUG] ブラウザを閉じました。')


# 実行: race_info を実際の受付中レースに書き換えてから呼ぶ
# debug_run_browser()


## メインループ — 監視 + 自動投票

新しい T-3 ファイルを検出するたびに信号を計算し、SPAT4 へ投票する。

**起動前チェックリスト:**
1. `config/spat4_credentials.json` の `dry_run` が `true` になっていることを確認 (初回は必須)
2. SPAT4 残高に投票可能な金額があるか確認
3. `MAX_TOTAL_BETS_PER_DAY` がリスク許容範囲内か確認

In [8]:
# ════════════════════════════════════════════════════════════
# メインループ
# ════════════════════════════════════════════════════════════
import sys as _sys
from IPython.display import clear_output as _clear_output

MAX_CONSECUTIVE_ERRORS = 3
MAX_T3_AGE_SEC = 300
MAX_BROWSER_RESTARTS = 3
STDOUT_CLEAR_INTERVAL_SEC = 120   # この秒数ごとに Jupyter セル出力をクリア


class _TeeStream:
    """stdout を画面 (Jupyter) とログファイルの両方に書き出す。"""
    def __init__(self, filepath, original):
        self._file = open(filepath, 'a', encoding='utf-8', buffering=1)
        self._orig = original

    def write(self, text):
        self._orig.write(text)
        self._file.write(text)

    def flush(self):
        self._orig.flush()
        try:
            self._file.flush()
        except Exception:
            pass

    def close(self):
        try:
            self._file.close()
        except Exception:
            pass


def _start_browser(sess: spat4.Spat4Session) -> object:
    driver = sess.new_driver()
    if sess.login(driver):
        return driver
    try:
        driver.quit()
    except Exception:
        pass
    return None


def main_loop():
    print(f'=== L905 自動投票監視 開始  {datetime.now():%Y-%m-%d %H:%M:%S} ===')
    print(f'    STAKE={STAKE_YEN}円  上限={MAX_TOTAL_BETS_PER_DAY}/day')
    print(f'    T3ファイル有効期限: {MAX_T3_AGE_SEC}秒 ({MAX_T3_AGE_SEC//60}分) 以内')

    # ── stdout → ログファイル Tee ──
    stdout_log = LOG_DIR / f'L905_stdout_{date.today():%Y%m%d}.log'
    _orig_stdout = _sys.stdout
    tee = _TeeStream(stdout_log, _orig_stdout)
    _sys.stdout = tee
    print(f'    stdout ログ: {stdout_log}')

    processed_race_ids = set()
    log_path = LOG_DIR / f'L905_bet_log_{date.today():%Y%m%d}.csv'
    if log_path.exists():
        try:
            df_log = pd.read_csv(log_path, encoding='utf-8-sig')
            processed_race_ids.update(df_log['race_id'].astype(str).unique())
            daily_counter.count = len(df_log)
            print(f'    log復元: 処理済み {len(processed_race_ids)} race / 累計投票 {daily_counter.count}')
        except Exception as e:
            print(f'    [WARN] log復元失敗: {e}')

    sess = spat4.Spat4Session(config, headless=False)
    consecutive_errors = 0
    browser_restarts = 0
    last_clear_ts = time.time()

    driver = _start_browser(sess)
    if driver is None:
        print('  [FATAL] SPAT4 login 失敗、終了')
        _sys.stdout = _orig_stdout
        tee.close()
        return

    try:
        while True:
            # ── 定期クリア (Jupyter セル出力のメモリ解放) ──
            if time.time() - last_clear_ts > STDOUT_CLEAR_INTERVAL_SEC:
                _clear_output(wait=True)
                print(f'=== L905 稼働中  累計投票 {daily_counter.count}  {datetime.now():%H:%M:%S} ==='
                      f'  (stdout → {stdout_log.name})')
                last_clear_ts = time.time()

            # ── 結果待ちレースの的中チェック ──
            still_pending = []
            for p in pending_hits:
                done = check_hit_and_notify(p)
                if not done:
                    still_pending.append(p)
            pending_hits[:] = still_pending

            # ── 新規 T3 ファイルをスキャン ──
            today_str = datetime.now().strftime('%Y%m%d')
            now_ts = time.time()
            t3_files = [
                p for p in SNAPSHOT_DIR.glob(f'*_T{T_END[1:]}_*_{today_str}-*.csv')
                if now_ts - p.stat().st_mtime <= MAX_T3_AGE_SEC
            ]

            for p in t3_files:
                rid = p.name.split('_')[0]
                if rid in processed_race_ids:
                    continue
                race_info = resolve_race_info(rid)
                t3_dt = _parse_t3_snapshot_dt(p)
                if t3_dt:
                    race_info['t3_snapshot_dt'] = t3_dt

                signals = compute_signals_for_race(rid)
                n_tan, n_uma = len(signals['tan_bets']), len(signals['uma_bets'])
                if n_tan + n_uma == 0:
                    print(f'[{datetime.now():%H:%M:%S}] {rid} {race_info["venue"]} R{race_info["race_num"]}: シグナルなし')
                else:
                    print(f'[{datetime.now():%H:%M:%S}] {rid} {race_info["venue"]} R{race_info["race_num"]}: '
                          f'シグナル 単勝{n_tan} 馬連{n_uma}')
                    try:
                        result = execute_bets_for_race(sess, driver, race_info, signals)
                    except spat4.BrowserSessionError as e:
                        print(f'  [WARN] ブラウザセッション切断 ({e.__class__.__name__}) — 再起動します')
                        try:
                            driver.quit()
                        except Exception:
                            pass
                        browser_restarts += 1
                        if browser_restarts > MAX_BROWSER_RESTARTS:
                            msg = f'ブラウザ再起動が {browser_restarts} 回に達したため停止します。'
                            print(f'  [FATAL] {msg}')
                            send_email_if_configured('[NAR L905] ブラウザ再起動上限で停止', msg)
                            return
                        driver = _start_browser(sess)
                        if driver is None:
                            msg = 'ブラウザ再起動後のログインに失敗しました。'
                            print(f'  [FATAL] {msg}')
                            send_email_if_configured('[NAR L905] 再ログイン失敗で停止', msg)
                            return
                        print(f'  [OK] ブラウザ再起動完了 ({browser_restarts}/{MAX_BROWSER_RESTARTS}回目)')
                        consecutive_errors = 0
                        processed_race_ids.add(rid)
                        continue

                    if result['ok']:
                        consecutive_errors = 0
                    else:
                        consecutive_errors += 1
                        print(f'  [WARN] 連続エラー {consecutive_errors}/{MAX_CONSECUTIVE_ERRORS}')
                        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                            msg = (f'連続 {consecutive_errors} 回の投票エラーが発生したため停止しました。\n'
                                   f'最後のエラー: {result["detail"]}')
                            print(f'  [FATAL] {msg}')
                            send_email_if_configured('[NAR L905] 連続エラーで停止', msg)
                            return
                processed_race_ids.add(rid)

            time.sleep(POLL_INTERVAL_SEC)
    except KeyboardInterrupt:
        print(f'[{datetime.now():%H:%M:%S}] Ctrl+C で停止')
    except Exception as e:
        print(f'[FATAL] {e}')
        traceback.print_exc()
    finally:
        try:
            driver.quit()
        except Exception:
            pass
        print(f'=== L905 終了  累計投票 {daily_counter.count} ===')
        _sys.stdout = _orig_stdout
        tee.close()


In [9]:
# # ════════════════════════════════════════════════════════════
# # DEBUG テスト — 浦和 12R 固定買い目で確認画面の手前まで
# # ════════════════════════════════════════════════════════════
# # 単勝  7        100円
# # 馬連  3-7      100円
# # 馬連  7-10     100円

# debug_run_browser(
#     race_info={
#         'race_id':   'DEBUG',
#         'venue':     '浦和',
#         'race_num':  12,
#         'race_date': datetime.now().strftime('%Y-%m-%d'),
#         'place_id':  '18',   # 浦和 JOCD
#     },
#     tan_umabans=[7],
#     uma_pairs=[(3, 7), (7, 10)],   # 馬連は小さい番号を P1 に
#     wait_sec=120,
# )


In [ ]:
# ════════════════════════════════════════════════════════════
# 【監視起動】 main_loop() を実行する (Ctrl+C で停止)
# DEBUG テストをしたい場合はこのセルを実行せず、下の Cell 10 を実行する
# ════════════════════════════════════════════════════════════
main_loop()


=== L905 自動投票監視 開始  2026-06-26 13:48:13 ===
    STAKE=100円  上限=500/day
    T3ファイル有効期限: 300秒 (5分) 以内
    stdout ログ: C:\Users\ppny9\workspace\nar\logs\L905_stdout_20260626.log
    log復元: 処理済み 1 race / 累計投票 1
  [INFO] トップページ: url=https://www.spat4.jp/keiba/pc  windows=1
  [INFO] ログインページ: url=https://www.spat4.jp/keiba/pc?HANDLERR=P000S  title=SPAT4インターネット投票
  [INFO] ログインフォームを検出 — 入力を開始します
  [INFO] 余分なタブを閉じました: SPAT4インターネット投票サービス FAQ
  [INFO] かんたんログインタブをクリック
  [WARN] 暗証番号フィールド未検出 — 利用可能フィールド: [('', 'hidden'), ('', 'hidden'), ('', 'hidden'), ('kantanLoginMessage', 'hidden'), ('HANDLERR', 'hidden'), ('LOGINR', 'hidden'), ('BEFOREHANDLERR', 'hidden'), ('K_LOGIN_INFO', 'hidden'), ('LOGIN_KIND', 'hidden'), ('NUM6FLG', 'hidden'), ('Sec-CH-UA-Platform', 'hidden'), ('Sec-CH-UA-Platform-Version', 'hidden'), ('Sec-CH-UA-Mobile', 'hidden'), ('Sec-CH-UA-Model', 'hidden'), ('Sec-CH-UA-Full-Version-List', 'hidden'), ('MEMBERNUMR', 'text'), ('MEMBERIDR', 'password'), ('BSLI', 'checkbox')]
  [OK] SPAT4 lo